# 02 — Análisis clásico del experimento (ATE)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import GROUP_LABELS, load_data

sns.set_theme(style='whitegrid')
df = load_data(ROOT / 'data' / 'datos_prueba_tecnica.csv')

## Funciones auxiliares

In [ ]:
def ate_proportion(df, treatment, control, outcome):
    p_t = df.loc[df['grupo'] == treatment, outcome].mean()
    p_c = df.loc[df['grupo'] == control, outcome].mean()
    diff = p_t - p_c
    n_t = (df['grupo'] == treatment).sum()
    n_c = (df['grupo'] == control).sum()
    se = np.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)
    ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se
    table = pd.crosstab(df.loc[df['grupo'].isin([treatment, control]), 'grupo'],
                        df.loc[df['grupo'].isin([treatment, control]), outcome])
    chi2, p, _, _ = stats.chi2_contingency(table)
    return {
        'comparison': f'{treatment} vs {control}',
        'outcome': outcome,
        'rate_treatment': p_t,
        'rate_control': p_c,
        'ate_pp': diff,
        'lift_pct': (diff / p_c * 100) if p_c else np.nan,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'p_value': p,
    }

comparisons = [('trat1', 'ctrl'), ('trat2', 'ctrl'), ('trat2', 'trat1')]
results = []
for outcome in ['or', 'ctor']:
    for t, c in comparisons:
        results.append(ate_proportion(df, t, c, outcome))
ate_df = pd.DataFrame(results).round(4)
ate_df

## Forest plot de efectos

In [ ]:
plot_df = ate_df.copy()
plot_df['label'] = plot_df['comparison'] + ' | ' + plot_df['outcome']
fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(plot_df))
ax.errorbar(plot_df['ate_pp'], y,
            xerr=[plot_df['ate_pp'] - plot_df['ci_low'], plot_df['ci_high'] - plot_df['ate_pp']],
            fmt='o', capsize=4)
ax.axvline(0, color='gray', linestyle='--')
ax.set_yticks(y)
ax.set_yticklabels(plot_df['label'])
ax.set_xlabel('ATE (puntos porcentuales)')
ax.set_title('Efectos promedio del tratamiento con IC 95%')
plt.tight_layout()
plt.show()

## Regresión logística ajustada por covariables

In [ ]:
formula = (
    'Q("or") ~ C(grupo, Treatment(reference="ctrl")) + edad + sexo + inve + '
    'uso_app + tarjeta_debito + C(tipo_tarjeta) + C(formacion)'
)
model_or = smf.logit(formula, data=df).fit(disp=0)
print(model_or.summary2().tables[1].loc[
    [x for x in model_or.params.index if 'grupo' in x]
])

In [ ]:
formula_ctor = formula.replace('Q("or") ~', 'ctor ~')
model_ctor = smf.logit(formula_ctor, data=df).fit(disp=0)
print(model_ctor.summary2().tables[1].loc[
    [x for x in model_ctor.params.index if 'grupo' in x]
])

**Conclusión:** Ambos nudges aumentan significativamente open rate y click rate. Trat2 supera a trat1 en clics. Los efectos persisten tras ajustar por covariables.